# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)
import os

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [4]:
# Some lists!

todos = []
completed = []

In [ ]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n". # Give a strike for True value in completed list
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [6]:
get_todo_report()

''

In [7]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [9]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [13]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [14]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [15]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [18]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)      ## This is the same as using if to check for tool_name and call the function -- refer to lab4
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [17]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="openai/gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [19]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [20]:
todos, completed = [], []
loop(messages)

Todo #1: Interpret problem setup and assumptions (distance between Boston and New York, same track, moving toward 
each other).
Todo #2: Estimate/choose a reasonable Boston–New York distance for calculation since not provided.
Todo #3: Set up relative motion equation accounting for 1-hour head start and solve for meeting time.
Todo #4: Compute meeting time in clock time and verify reasonableness.
Todo #5: Present final answer clearly in Rich console markup.

Assumed trains travel directly toward each other along the BostonNew York corridor on the same line. Let D be the 
distance between the cities. Boston train departs 2:00 pm at 60 mph; New York train departs 3:00 pm at 80 mph 
toward Boston. They meet when total distances covered equals D.

Todo #1: Interpret problem setup and assumptions (distance between Boston and New York, same track, moving toward 
each other).
Todo #2: Estimate/choose a reasonable Boston–New York distance for calculation since not provided.
Todo #3: Set up relative motion equation accounting for 1-hour head start and solve for meeting time.
Todo #4: Compute meeting time in clock time and verify reasonableness.
Todo #5: Present final answer clearly in Rich console markup.

Distance not given; used a reasonable rail/road corridor estimate BostonNew York 180 miles (typical driving 
distance ~215 mi; straight-line ~190 mi; rail often ~230 mi). Chose 200 miles as a round, reasonable midpoint to 
produce a concrete meeting time.

Todo #1: Interpret problem setup and assumptions (distance between Boston and New York, same track, moving toward 
each other).
Todo #2: Estimate/choose a reasonable Boston–New York distance for calculation since not provided.
Todo #3: Set up relative motion equation accounting for 1-hour head start and solve for meeting time.
Todo #4: Compute meeting time in clock time and verify reasonableness.
Todo #5: Present final answer clearly in Rich console markup.

...

Meeting time = 3:00 pm + t = 3:00 pm + 1:00 = 4:00 pm. Check: By 4:00 pm Boston train traveled 2 hours at 60=120 
miles; NY train traveled 1 hour at 80=80 miles; total 200 miles, matches assumed D.

Todo #1: Interpret problem setup and assumptions (distance between Boston and New York, same track, moving toward 
each other).
Todo #2: Estimate/choose a reasonable Boston–New York distance for calculation since not provided.
Todo #3: Set up relative motion equation accounting for 1-hour head start and solve for meeting time.
Todo #4: Compute meeting time in clock time and verify reasonableness.
Todo #5: Present final answer clearly in Rich console markup.

Prepared final statement including the general formula t=(D-60)/140 and the computed time using the chosen estimate
D=200 miles.

Todo #1: Interpret problem setup and assumptions (distance between Boston and New York, same track, moving toward 
each other).
Todo #2: Estimate/choose a reasonable Boston–New York distance for calculation since not provided.
Todo #3: Set up relative motion equation accounting for 1-hour head start and solve for meeting time.
Todo #4: Compute meeting time in clock time and verify reasonableness.
Todo #5: Present final answer clearly in Rich console markup.

Assuming a reasonable Boston–New York distance of 200 miles:

• By 3:00 pm, the Boston train (60 mph) has gone 60 miles.  
• Remaining distance then: 200 − 60 = 140 miles.  
• After 3:00 pm they close at 60 + 80 = 140 mph.  
• Time to meet after 3:00 pm: 140 / 140 = 1 hour.

They meet at 4:00 pm.

General form (if the city distance is D miles): time after 3:00 pm is (D − 60)/140 hours.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>